# Clase 10 — Práctica: Rotosolve y optimizadores

En la clase comparaste Rotosolve, Nelder-Mead y SPSA sobre un Hamiltoniano de dos términos. En esta práctica añades un tercer término y compruebas si las conclusiones se mantienen.

## Enunciado

Añade un tercer término al Hamiltoniano de ejemplo: `["y", "x"]` con peso `0.4` (recuerda actualizar tanto `terminos_hamiltoniano` como `pesos_hamiltoniano`). Vuelve a ejecutar los tres optimizadores y anota, para cada uno, la energía final obtenida. Comprueba si Rotosolve sigue siendo competitivo frente a Nelder-Mead y SPSA ahora que el Hamiltoniano tiene un término más.

**Pistas**

- Reutiliza `capa_ansatz`, `rotar_a_base_de_medida`, `valor_esperado` y `rotosolve` tal cual
- Solo cambia `terminos_hamiltoniano` y `pesos_hamiltoniano`; `num_parametros` no cambia, porque depende del número de qubits y de capas, no del número de términos
- Ejecuta los tres optimizadores sobre la función `energia` ya actualizada y guarda cada resultado en una variable distinta para poder compararlos al final

In [ ]:
!pip install -q cirq scipy noisyopt

In [ ]:
import numpy as np
import random
import cirq
from scipy.optimize import minimize
from noisyopt import minimizeSPSA


def rotosolve(funcion_energia, num_parametros, iteraciones=30):
    """Optimiza cada parámetro de forma analítica, ciclo a ciclo."""
    theta = np.random.uniform(-np.pi, np.pi, num_parametros)
    for _ in range(iteraciones):
        for d in range(num_parametros):
            phi = random.uniform(-np.pi, np.pi)
            theta_prueba = np.copy(theta)

            theta_prueba[d] = phi
            energia_phi = funcion_energia(theta_prueba)
            theta_prueba[d] = phi + np.pi / 2
            energia_mas = funcion_energia(theta_prueba)
            theta_prueba[d] = phi - np.pi / 2
            energia_menos = funcion_energia(theta_prueba)

            theta[d] = phi - np.pi / 2 - np.arctan2(
                2 * energia_phi - energia_mas - energia_menos, energia_mas - energia_menos
            )
    return {"fun": funcion_energia(theta), "params": theta}


def capa_ansatz(circuito, qubits, parametros):
    for qubit in qubits:
        circuito.append([cirq.rx(parametros[0]).on(qubit)])
        circuito.append([cirq.rz(parametros[1]).on(qubit)])
    for i in range(len(qubits) - 1):
        circuito.append([cirq.CNOT(qubits[i], qubits[i + 1])])
    return circuito


def rotar_a_base_de_medida(circuito, qubits, terminos_pauli):
    for i, qubit in enumerate(qubits):
        if terminos_pauli[i] == "x":
            circuito.append(cirq.ry(-np.pi / 2).on(qubit))
        elif terminos_pauli[i] == "y":
            circuito.append(cirq.rx(np.pi / 2).on(qubit))
    return circuito


def valor_esperado(circuito, qubits_medidos, repeticiones=1000):
    if len(qubits_medidos) == 0:
        return 1
    circuito.append(cirq.measure(*qubits_medidos, key="resultado"))
    simulador = cirq.Simulator()
    resultados = simulador.run(circuito, repetitions=repeticiones)
    medidas = resultados.measurements["resultado"]
    return np.mean([1 if sum(fila) % 2 == 0 else -1 for fila in medidas])

In [ ]:
# Hamiltoniano original de dos términos
terminos_hamiltoniano = [["x", "z"], ["z", "y"]]
pesos_hamiltoniano = [0.6, 0.3]
num_qubits, num_capas = 2, 2

# TODO: añade un tercer término ["y", "x"] con peso 0.4 a terminos_hamiltoniano y a pesos_hamiltoniano


def energia(parametros):
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    total = 0
    for termino, peso in zip(terminos_hamiltoniano, pesos_hamiltoniano):
        circuito = cirq.Circuit()
        for _ in range(num_capas):
            circuito = capa_ansatz(circuito, qubits, parametros)
        circuito = rotar_a_base_de_medida(circuito, qubits, termino)
        qubits_a_medir = [qubits[i] for i in range(len(termino)) if termino[i] != "i"]
        total += peso * valor_esperado(circuito, qubits_a_medir)
    return total


num_parametros = num_qubits * num_capas

In [ ]:
# Las tres funciones de optimización ya están listas para ejecutarse sobre la energía actualizada
resultado_nm = minimize(energia, np.random.uniform(0, 2 * np.pi, num_parametros), method="Nelder-Mead")
resultado_spsa = minimizeSPSA(energia, bounds=[[0, 2 * np.pi]] * num_parametros,
                               x0=np.random.uniform(0, 2 * np.pi, num_parametros), niter=100, paired=False)
resultado_roto = rotosolve(energia, num_parametros)

# TODO: imprime la energía final ("fun") de resultado_nm, resultado_spsa y resultado_roto

# TODO: anota en un comentario cuál de los tres optimizadores obtuvo la energía más baja
# con el Hamiltoniano de tres términos

<details>
<summary>💡 Ver solución</summary>

```python
# Hamiltoniano con el tercer término añadido: ["y", "x"] con peso 0.4
terminos_hamiltoniano = [["x", "z"], ["z", "y"], ["y", "x"]]
pesos_hamiltoniano = [0.6, 0.3, 0.4]
num_qubits, num_capas = 2, 2


def energia(parametros):
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    total = 0
    for termino, peso in zip(terminos_hamiltoniano, pesos_hamiltoniano):
        circuito = cirq.Circuit()
        for _ in range(num_capas):
            circuito = capa_ansatz(circuito, qubits, parametros)
        circuito = rotar_a_base_de_medida(circuito, qubits, termino)
        qubits_a_medir = [qubits[i] for i in range(len(termino)) if termino[i] != "i"]
        total += peso * valor_esperado(circuito, qubits_a_medir)
    return total


num_parametros = num_qubits * num_capas

resultado_nm = minimize(energia, np.random.uniform(0, 2 * np.pi, num_parametros), method="Nelder-Mead")
resultado_spsa = minimizeSPSA(energia, bounds=[[0, 2 * np.pi]] * num_parametros,
                               x0=np.random.uniform(0, 2 * np.pi, num_parametros), niter=100, paired=False)
resultado_roto = rotosolve(energia, num_parametros)

print("Nelder-Mead:", resultado_nm["fun"])
print("SPSA:", resultado_spsa["fun"])
print("Rotosolve:", resultado_roto["fun"])

# Con un término más en el Hamiltoniano, compara las tres energías finales impresas arriba:
# normalmente siguen siendo parecidas entre sí, porque los tres optimizadores buscan
# el mismo mínimo, pero Rotosolve puede necesitar alguna pasada adicional para
# estabilizarse, porque ahora hay un término más que influye en la forma de la onda
# de energía de cada parámetro.
```

</details>